# Database overview notebook

> First time use: follow instructions in the README.md file in this directory.

In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook(kleio_image="kleio-server", kleio_version="12.9.592")
tlnb.print_info()


## Ligar à base de dados

---

## Connect to database

In [ ]:
from timelink.api.database import TimelinkDatabase

db: TimelinkDatabase = tlnb.db

## Database status

In [ ]:
tlnb.table_row_count_df()

## Recent sources

In [ ]:
number_of_sources = 100

Source = db.get_model('source')

with db.session() as session:
    sources = session.query(Source).order_by(Source.updated.desc()).all()
    for source in sources[:number_of_sources]:
        print(f"{source.id:32} {source.updated:%Y-%m-%d %H:%M:%S} {source.kleiofile}")


### Tipos de attributos

---

### Attribute types

In [ ]:
import pandas as pd

from sqlalchemy import func
from sqlalchemy import select


pd.set_option('display.max_rows', 500)

attr_table = tlnb.db.get_table('attributes')
tlnb.db.describe('attributes', show=True)
print()
stmt = select(
    attr_table.c.the_type,
    func.count().label('count'),
    func.count(func.distinct(attr_table.c.the_value)).label('distinct_value')
    ).group_by('the_type')
print(stmt)
print()

with tlnb.db.session() as session:
    # nml2 = session.query(Attribute.the_type,func.count().label('tot')).group_by(Attribute.the_type).all()
    nml = session.execute(stmt)
    attribute_df = pd.DataFrame(nml)

attribute_df

## Vocabulary of attributes

In [ ]:
from timelink.pandas import attribute_values

df_totals = attribute_values('jesuita-fundador',db=tlnb.db)
df_totals

## Mostrar uma fonte

---

## Show a source

In [ ]:
id = 'dehergne-locations-1644'
with db.session() as session:
    source = session.get(Source,id)
    print(source.to_kleio())

## Obter modelos para acesso aos dados

---

## Obtain models for data access

In [ ]:
db.get_models_ids()


In [ ]:
Model = db.get_model('geoentity')

with db.session() as session:
    entities = session.query(Model).all()
    for entity in entities[:10]:
        print(entity.to_kleio())
        print()